# Vocal F0 Pipeline Visualizer

This notebook walks through every stage of the `audio2midi` vocal transcription pipeline,
showing what each technique does to the pitch signal and why.

## Pipeline Stages

1. **Raw F0 extraction** — RMVPE outputs a continuous Hz contour at 10 ms / frame
2. **F0 median filter** — per-voiced-segment smoothing suppresses vibrato before quantization
3. **Semitone snapping** — continuous Hz → discrete MIDI note numbers
4. **Min-duration filter** — discard segments shorter than a 32nd note
5. **Silence-aware merge** — collapse ornament fragments (A-B-A patterns, micro-stutters)
6. **Same-pitch merge** — collapse consecutive same-pitch notes separated by ≤ 1 eighth note
7. **Beat-gated onset split** — re-split long notes at beat-aligned syllable onsets
8. **Snap-to-beats** *(optional)* — quantize note boundaries to nearest 16th-note grid

**Source implementation:** `audio2midi/transcribers/rmvpe.py`  
**Research session:** `research/2026-04-18_f0-median-filter-research/notes.md`

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import librosa
from pathlib import Path
from scipy.ndimage import median_filter
from scipy.signal import butter, sosfilt, find_peaks

plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (14, 4),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

# --- Configuration ---
# Paths — adjust to your local research session directory
REPO_ROOT       = Path('../../')                          # relative to notebook dir
VOCALS_WAV      = REPO_ROOT / 'research/2026-04-18_f0-median-filter-research/stems/htdemucs/mix/vocals.wav'
CLICK_WAV       = REPO_ROOT / 'research/2026-04-18_f0-median-filter-research/f0_raw_mix.wav'  # used for beat times only
RMVPE_CKPT      = REPO_ROOT / 'research/2026-04-17_vocal-pitch-tracker-research/rmvpe.pt'

# RMVPE inference constants [source-code] audio2midi/transcribers/rmvpe.py
TARGET_SR       = 16000          # [source-code] rmvpe.py:_TARGET_SR
HOP_SAMPLES     = 160            # [source-code] rmvpe.py:_HOP_S = 160/16000
HOP_S           = HOP_SAMPLES / TARGET_SR   # 0.010 s per frame [back-calc]

# Tempo / rhythm [measured] bpm_detect on raw mix, research/2026-04-18_f0-median-filter-research/notes.md
BPM             = 152.05         # [measured]
BEAT_S          = 60.0 / BPM    # [back-calc]
MIN_NOTE_S      = BEAT_S / 8.0  # 32nd note floor [back-calc]
SIXTEENTH_S     = BEAT_S / 4.0  # [back-calc]
EIGHTH_S        = BEAT_S / 2.0  # [back-calc]

# F0 median filter [measured] research/2026-04-18_f0-median-filter-research/notes.md
F0_FILTER_FRAMES = 7             # [measured] 70 ms; vibrato period ~195 ms, window = 0.36x period

# Onset detection [source-code] rmvpe.py
ONSET_HOP       = 256            # [source-code] rmvpe.py:_ONSET_HOP
ONSET_DELTA     = 0.07           # [source-code] rmvpe.py:_ONSET_DELTA

# Beat-gated split tolerance
BEAT_TOLERANCE_FRAC = 0.25       # [source-code] rmvpe.py — 25% of beat interval

# Visualisation: time window to plot (seconds)
VIZ_START       = 4.0            # [assumed] shows an interesting melodic phrase
VIZ_END         = 20.0           # [assumed]

# Add repo to path for RMVPE import
sys.path.insert(0, str(REPO_ROOT.resolve()))

print('Configuration loaded.')
print(f'  BPM={BPM}, min_note={MIN_NOTE_S*1000:.1f}ms, 16th={SIXTEENTH_S*1000:.1f}ms, 8th={EIGHTH_S*1000:.1f}ms')
print(f'  F0 filter: {F0_FILTER_FRAMES} frames = {F0_FILTER_FRAMES*10} ms')

## 1. Raw F0 Extraction

RMVPE (Robust Model for Vocal Pitch Estimation) runs on the separated vocals stem at
16 kHz with a 10 ms hop. It outputs a continuous Hz value per frame: positive when
the model detects voiced pitch, zero for silence or unvoiced frames.

The raw contour captures everything — melody, vibrato, portamento, and quantization
noise from the model's internal frame-level decisions. The goal of subsequent stages
is to turn this noisy continuous signal into clean discrete notes.

**Key parameters:** `thred=0.03` (voiced/unvoiced confidence threshold) `[source-code]`

In [ ]:
# Load audio and run RMVPE
y_vocals, _ = librosa.load(str(VOCALS_WAV), sr=TARGET_SR, mono=True)

from audio2midi.transcribers._rmvpe_impl import RMVPE0Predictor
model = RMVPE0Predictor(str(RMVPE_CKPT), device='cpu')
f0_raw = model.infer_from_audio(y_vocals, thred=0.03)  # thred [source-code]

times = np.arange(len(f0_raw)) * HOP_S
voiced_mask = f0_raw > 0

print(f'F0 frames: {len(f0_raw)}  ({len(f0_raw)*HOP_S:.1f}s)')
print(f'Voiced frames: {voiced_mask.sum()} ({100*voiced_mask.mean():.1f}%)')
print(f'F0 range: {f0_raw[voiced_mask].min():.1f} – {f0_raw[voiced_mask].max():.1f} Hz')

In [ ]:
# Helpers for converting Hz to note names and MIDI numbers
def hz_to_midi(hz):
    """Convert Hz to MIDI note number (float)."""
    return 12 * np.log2(np.maximum(hz, 1e-6) / 440.0) + 69

def midi_to_note_name(midi_int):
    names = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
    return f"{names[midi_int % 12]}{midi_int // 12 - 1}"

# Convert voiced F0 to MIDI float for plotting
f0_midi_float = np.where(voiced_mask, hz_to_midi(f0_raw), np.nan)

mask_viz = (times >= VIZ_START) & (times <= VIZ_END)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(times[mask_viz], f0_midi_float[mask_viz], color='steelblue', lw=0.8, label='Raw F0 (MIDI)')
ax.set_title('Stage 1 — Raw RMVPE F0 Contour')
ax.set_xlabel('Time (s)')
ax.set_ylabel('MIDI note')
ax.set_yticks(range(57, 84, 2))
ax.set_yticklabels([midi_to_note_name(m) for m in range(57, 84, 2)])
ax.legend()
plt.tight_layout()
plt.show()

## 2. F0 Median Filter (Vibrato Suppression)

A median filter with a window of **7 frames (70 ms)** is applied per-voiced-segment
to the Hz contour *before* semitone snapping. This smooths rapid pitch oscillations
(vibrato, portamento jitter) in the continuous domain, so the discretization step
sees a stable pitch rather than an oscillation that would produce alternating semitone notes.

**Why median, not mean?** The median preserves sharp pitch steps (genuine note changes)
while averaging out sustained oscillations — a mean filter would smear step boundaries.

**Why 70 ms?** Measured vibrato rate on this track = 5.14 Hz → period = 194.5 ms.
70 ms = 0.36× the period, safely below the half-period where step smearing begins.
*(Nix et al., Journal of Voice 30(6), 2016: typical range 4.5–6.5 Hz)* `[measured]`

The filter is applied **per contiguous voiced segment** — never across silence boundaries.

In [ ]:
def apply_f0_median_filter(f0_hz, window_frames):
    """Apply median filter per voiced segment. Silence frames are untouched."""
    if window_frames <= 1:
        return f0_hz.copy()
    result = f0_hz.copy().astype(float)
    voiced = f0_hz > 0
    # Detect segment boundaries via first-difference of voiced mask
    changes = np.diff(voiced.astype(int), prepend=0, append=0)
    starts = np.where(changes == 1)[0]
    ends   = np.where(changes == -1)[0]
    for s, e in zip(starts, ends):
        if e - s >= window_frames:  # only filter segments longer than the window
            result[s:e] = median_filter(f0_hz[s:e].astype(float), size=window_frames)
    return result

f0_filtered = apply_f0_median_filter(f0_raw, F0_FILTER_FRAMES)
f0_filtered_midi = np.where(f0_filtered > 0, hz_to_midi(f0_filtered), np.nan)

# Measure how much the filter moved the pitch
voiced_both = voiced_mask & (f0_filtered > 0)
delta_cents = 1200 * np.log2(np.maximum(f0_filtered[voiced_both], 1e-6) /
                              np.maximum(f0_raw[voiced_both], 1e-6))
print(f'Filter effect (voiced frames only):')
print(f'  Mean |Δ| = {np.abs(delta_cents).mean():.1f} cents')
print(f'  Max  |Δ| = {np.abs(delta_cents).max():.1f} cents')
print(f'  Frames moved >50 cents: {(np.abs(delta_cents) > 50).sum()} ({100*(np.abs(delta_cents)>50).mean():.1f}%)')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(times[mask_viz], f0_midi_float[mask_viz], color='steelblue', lw=0.8, label='Raw F0', alpha=0.8)
axes[0].plot(times[mask_viz], f0_filtered_midi[mask_viz], color='tomato', lw=1.2, label=f'Filtered ({F0_FILTER_FRAMES}fr / {F0_FILTER_FRAMES*10}ms)')
axes[0].set_title('Stage 2 — F0 Median Filter: Raw vs Filtered')
axes[0].set_ylabel('MIDI note')
axes[0].set_yticks(range(57, 84, 2))
axes[0].set_yticklabels([midi_to_note_name(m) for m in range(57, 84, 2)])
axes[0].legend()

# Zoom: 2-second window showing vibrato smoothing
ZOOM_S, ZOOM_E = 12.0, 14.0
zoom_mask = (times >= ZOOM_S) & (times <= ZOOM_E)
axes[1].plot(times[zoom_mask], f0_midi_float[zoom_mask], color='steelblue', lw=1.0, label='Raw F0', alpha=0.8)
axes[1].plot(times[zoom_mask], f0_filtered_midi[zoom_mask], color='tomato', lw=1.5, label=f'Filtered')
axes[1].set_title(f'Zoom: {ZOOM_S}–{ZOOM_E}s — vibrato smoothing detail')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('MIDI note')
axes[1].set_yticks(range(57, 84, 2))
axes[1].set_yticklabels([midi_to_note_name(m) for m in range(57, 84, 2)])
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Semitone Snapping + Min-Duration Filter

The filtered Hz contour is snapped to the nearest MIDI semitone using the formula:

```
midi = round(12 × log₂(f0 / 440) + 69)
```

Frames outside the vocal range C2–C6 (MIDI 36–84) are treated as unvoiced.

The frame sequence is then segmented into notes: each run of identical semitone
values becomes one note. Segments shorter than a **32nd note** (= `60/BPM/8` seconds)
are discarded as sub-rhythmic noise. `[source-code] rmvpe.py`

In [ ]:
MIDI_LO = round(hz_to_midi(librosa.note_to_hz('C2')))  # 36  [source-code]
MIDI_HI = round(hz_to_midi(librosa.note_to_hz('C6')))  # 84  [source-code]

def semitone_snap(f0_hz):
    """Snap Hz contour to MIDI integers; unvoiced or out-of-range → -1."""
    voiced = (f0_hz > 0) & (f0_hz >= librosa.note_to_hz('C2')) & (f0_hz <= librosa.note_to_hz('C6'))
    midi = np.where(
        voiced,
        np.round(12 * np.log2(np.maximum(f0_hz, 1e-6) / 440.0) + 69).astype(int),
        -1,
    )
    return midi

def segment_notes(midi_track, times_arr, min_dur_s):
    """Segment midi_track into (start_s, end_s, pitch) tuples, dropping short segments."""
    raw = []
    in_note = False
    note_start = 0.0
    note_midi = 0
    for t, midi in zip(times_arr, midi_track):
        if midi >= 0:
            if not in_note:
                in_note = True; note_start = float(t); note_midi = int(midi)
            elif int(midi) != note_midi:
                if float(t) - note_start >= min_dur_s:
                    raw.append((note_start, float(t), note_midi))
                note_start = float(t); note_midi = int(midi)
        else:
            if in_note:
                if float(t) - note_start >= min_dur_s:
                    raw.append((note_start, float(t), note_midi))
                in_note = False
    return raw

midi_track = semitone_snap(f0_filtered)
notes_snap = segment_notes(midi_track, times, MIN_NOTE_S)

def count_unison(notes):
    return sum(1 for i in range(len(notes)-1) if notes[i][2] == notes[i+1][2])

print(f'After semitone snap + min-duration filter:')
print(f'  Notes: {len(notes_snap)},  unison repeats: {count_unison(notes_snap)}')

In [ ]:
def plot_notes(ax, notes, color, label, alpha=0.7):
    """Draw a piano-roll style note bar for each note."""
    for s, e, p in notes:
        ax.barh(p, e - s, left=s, height=0.6, color=color, alpha=alpha, edgecolor='none')
    ax.plot([], [], color=color, label=label, linewidth=6, alpha=alpha)  # legend proxy

fig, ax = plt.subplots(figsize=(14, 5))
plot_notes(ax, [(s,e,p) for s,e,p in notes_snap if VIZ_START <= s <= VIZ_END], 'steelblue', 'Notes after snap')
ax.set_title('Stage 3 — Semitone Snap + Min-Duration Filter (piano roll)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('MIDI note')
ax.set_yticks(range(57, 84, 2))
ax.set_yticklabels([midi_to_note_name(m) for m in range(57, 84, 2)])
ax.set_xlim(VIZ_START, VIZ_END)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Silence-Aware Merge

Two short-range merge patterns clean up micro-artifacts that survive semitone snapping:

- **A-B-A ornament** — three notes where the outer two match pitch (`n0 == n2`) and the
  middle B is short (< 1.5× min_note) and both gaps are tiny (< 0.5× min_note). The
  B is a pitch detector hiccup; collapse all three into one A.
- **Near-unison stutter** — two consecutive notes within 2 semitones where the second
  is short and the gap is tiny. Merge into one note at the first pitch.

Both patterns loop until no more merges are possible. `[source-code] rmvpe.py`

In [ ]:
def silence_aware_merge(raw, min_note_s):
    max_silence_s = min_note_s * 0.5   # max gap to bridge  [source-code]
    gap_s         = min_note_s * 1.5   # max middle-note duration to collapse  [source-code]
    changed = True
    while changed:
        changed = False
        out = []; i = 0
        while i < len(raw):
            # Pattern 1: A-B-A ornament
            if i + 2 < len(raw):
                s0,e0,n0 = raw[i]; s1,e1,n1 = raw[i+1]; s2,e2,n2 = raw[i+2]
                if (s1-e0 <= max_silence_s and s2-e1 <= max_silence_s
                        and n0 == n2 and (e1-s1) < gap_s):
                    out.append((s0, e2, n0)); i += 3; changed = True; continue
            # Pattern 2: near-unison stutter
            if i + 1 < len(raw):
                s0,e0,n0 = raw[i]; s1,e1,n1 = raw[i+1]
                if (s1-e0 <= max_silence_s and (e1-s1) < gap_s and abs(n1-n0) <= 2):
                    out.append((s0, e1, n0)); i += 2; changed = True; continue
            out.append(raw[i]); i += 1
        raw = out
    return raw

notes_sil = silence_aware_merge(notes_snap, MIN_NOTE_S)
print(f'After silence-aware merge:')
print(f'  Notes: {len(notes_sil)}  (dropped {len(notes_snap)-len(notes_sil)}),  unison: {count_unison(notes_sil)}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
plot_notes(axes[0], [(s,e,p) for s,e,p in notes_snap if VIZ_START<=s<=VIZ_END], 'steelblue', 'Before merge')
axes[0].set_title('Stage 4 — Silence-Aware Merge: Before')
axes[0].set_ylabel('MIDI note')
axes[0].set_yticks(range(57,84,2)); axes[0].set_yticklabels([midi_to_note_name(m) for m in range(57,84,2)])
axes[0].legend(); axes[0].set_xlim(VIZ_START, VIZ_END)

plot_notes(axes[1], [(s,e,p) for s,e,p in notes_sil if VIZ_START<=s<=VIZ_END], 'seagreen', 'After merge')
axes[1].set_title('Stage 4 — Silence-Aware Merge: After')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('MIDI note')
axes[1].set_yticks(range(57,84,2)); axes[1].set_yticklabels([midi_to_note_name(m) for m in range(57,84,2)])
axes[1].legend(); axes[1].set_xlim(VIZ_START, VIZ_END)
plt.tight_layout(); plt.show()

## 5. Same-Pitch Merge (×2)

After silence-aware merge, consecutive same-pitch notes still appear when a sustained
note is interrupted by a brief moment of silence or a one-frame pitch deviation.

**Pass 1** (before onset split): collapse any two adjacent same-pitch notes whose gap
is ≤ one eighth note (≈ 197 ms at 152 BPM), unless a beat-aligned time falls in the gap
(which would indicate a genuine syllable boundary).

**Pass 2** (after onset split): the onset splitter may re-cut merged notes; this
pass cleans up any same-pitch adjacencies it reintroduces.

The beat-guard preserves intentional repeated notes on beat boundaries. `[source-code] rmvpe.py`

In [ ]:
def nearest_beat_distance(t, beat_arr):
    """Return seconds to the nearest beat in beat_arr."""
    if len(beat_arr) == 0:
        return float('inf')
    idx = np.searchsorted(beat_arr, t)
    candidates = beat_arr[max(0, idx-1): idx+1]
    return float(np.min(np.abs(candidates - t)))

def same_pitch_merge(raw, max_gap_s, beat_arr, beat_tol):
    """Merge consecutive same-pitch notes with gap <= max_gap_s.
    Blocked if a beat falls inside the gap (genuine repeated syllable).
    """
    changed = True
    while changed:
        changed = False; out = []; i = 0
        while i < len(raw):
            if i + 1 < len(raw):
                s0,e0,n0 = raw[i]; s1,e1,n1 = raw[i+1]
                gap = s1 - e0
                if n0 == n1 and 0 <= gap <= max_gap_s:
                    # Check if any beat falls strictly inside the gap
                    beat_in_gap = len(beat_arr) > 0 and any(
                        nearest_beat_distance(t, beat_arr) <= beat_tol
                        for t in np.linspace(e0, s1, max(2, int(gap / 0.01)))
                        if e0 < t < s1
                    )
                    if not beat_in_gap:
                        out.append((s0, e1, n0)); i += 2; changed = True; continue
            out.append(raw[i]); i += 1
        raw = out
    return raw

# For this visualizer we run without beat times to show the merge in isolation
# (beat_arr=[] disables the guard so all same-pitch pairs within 8th note merge)
BEAT_ARR_EMPTY = np.array([], dtype=float)
notes_sp1 = same_pitch_merge(notes_sil, EIGHTH_S, BEAT_ARR_EMPTY, 0.0)
print(f'After same-pitch merge pass 1:')
print(f'  Notes: {len(notes_sp1)}  (dropped {len(notes_sil)-len(notes_sp1)}),  unison: {count_unison(notes_sp1)}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
plot_notes(axes[0], [(s,e,p) for s,e,p in notes_sil if VIZ_START<=s<=VIZ_END], 'seagreen', 'Before same-pitch merge')
axes[0].set_title('Stage 5 — Same-Pitch Merge: Before')
axes[0].set_ylabel('MIDI note')
axes[0].set_yticks(range(57,84,2)); axes[0].set_yticklabels([midi_to_note_name(m) for m in range(57,84,2)])
axes[0].legend(); axes[0].set_xlim(VIZ_START, VIZ_END)

plot_notes(axes[1], [(s,e,p) for s,e,p in notes_sp1 if VIZ_START<=s<=VIZ_END], 'darkorange', 'After same-pitch merge')
axes[1].set_title('Stage 5 — Same-Pitch Merge: After')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('MIDI note')
axes[1].set_yticks(range(57,84,2)); axes[1].set_yticklabels([midi_to_note_name(m) for m in range(57,84,2)])
axes[1].legend(); axes[1].set_xlim(VIZ_START, VIZ_END)
plt.tight_layout(); plt.show()

## 6. Beat-Gated Onset Split

Long held notes may span multiple syllables. Librosa onset detection finds syllable
boundaries on the vocals stem; these are used to split notes at genuine articulation points.

**Beat-gating:** a split is only permitted when the onset falls within 25% of a beat
interval from a known beat. This prevents the onset detector from re-introducing the
sub-16th-note fragments we just merged. `[source-code] rmvpe.py`

**Minimum split fragment:** a 16th note (= `60/BPM/4`). Any fragment shorter than this
is discarded even if the onset is beat-aligned.

In [ ]:
# Onset detection on native-sr vocals stem
y_native, sr_native = librosa.load(str(VOCALS_WAV), sr=None, mono=True)
onset_frames = librosa.onset.onset_detect(
    y=y_native, sr=sr_native,
    hop_length=ONSET_HOP,   # [source-code]
    delta=ONSET_DELTA,      # [source-code]
    backtrack=True,
)
onset_times_arr = librosa.frames_to_time(onset_frames, sr=sr_native, hop_length=ONSET_HOP)
print(f'Onsets detected: {len(onset_times_arr)} ({len(onset_times_arr)/len(f0_raw)/HOP_S:.2f}/s)')

# Simulate beat grid from BPM (in production this comes from bpm_detect click track)
# Here we build an idealized beat grid for visualisation purposes
beat_grid = np.arange(0, times[-1] + BEAT_S, BEAT_S)  # [back-calc] from measured BPM
beat_interval = BEAT_S
beat_tol = beat_interval * BEAT_TOLERANCE_FRAC  # [source-code]

SPLIT_MIN_S = max(MIN_NOTE_S, SIXTEENTH_S)  # [source-code]
margin = SPLIT_MIN_S * 0.5
onset_arr = np.sort(onset_times_arr)

split = []
for start_s, end_s, note in notes_sp1:
    interior = onset_arr[(onset_arr > start_s + margin) & (onset_arr < end_s - margin)]
    # Filter to beat-aligned onsets only
    interior = np.array([o for o in interior
                         if nearest_beat_distance(o, beat_grid) <= beat_tol])
    if len(interior) == 0:
        split.append((start_s, end_s, note))
    else:
        boundaries = [start_s] + list(interior) + [end_s]
        for a, b in zip(boundaries[:-1], boundaries[1:]):
            if b - a >= SPLIT_MIN_S:
                split.append((a, b, note))

notes_split = split

# Same-pitch merge pass 2 — clean up re-introduced fragments
notes_sp2 = same_pitch_merge(notes_split, EIGHTH_S, beat_grid, beat_tol)

print(f'After onset split: {len(notes_split)} notes  (added {len(notes_split)-len(notes_sp1)})')
print(f'After same-pitch merge pass 2: {len(notes_sp2)} notes  (dropped {len(notes_split)-len(notes_sp2)})')
print(f'Unison repeats: {count_unison(notes_sp2)}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

plot_notes(axes[0], [(s,e,p) for s,e,p in notes_sp1 if VIZ_START<=s<=VIZ_END], 'darkorange', 'Before onset split')
axes[0].set_title('Stage 6 — Beat-Gated Onset Split: Input')
axes[0].set_ylabel('MIDI note')
axes[0].set_yticks(range(57,84,2)); axes[0].set_yticklabels([midi_to_note_name(m) for m in range(57,84,2)])
axes[0].legend(); axes[0].set_xlim(VIZ_START, VIZ_END)

# Show onsets and beat grid
for ot in onset_times_arr[(onset_times_arr >= VIZ_START) & (onset_times_arr <= VIZ_END)]:
    axes[0].axvline(ot, color='gray', lw=0.5, alpha=0.5)
for bt in beat_grid[(beat_grid >= VIZ_START) & (beat_grid <= VIZ_END)]:
    axes[0].axvline(bt, color='red', lw=0.8, alpha=0.3)
axes[0].plot([], [], color='gray', lw=1, label='onset', alpha=0.5)
axes[0].plot([], [], color='red', lw=1, label='beat', alpha=0.5)
axes[0].legend()

plot_notes(axes[1], [(s,e,p) for s,e,p in notes_split if VIZ_START<=s<=VIZ_END], 'mediumpurple', 'After onset split')
axes[1].set_title('Stage 6 — After Beat-Gated Onset Split')
axes[1].set_ylabel('MIDI note')
axes[1].set_yticks(range(57,84,2)); axes[1].set_yticklabels([midi_to_note_name(m) for m in range(57,84,2)])
axes[1].legend(); axes[1].set_xlim(VIZ_START, VIZ_END)

plot_notes(axes[2], [(s,e,p) for s,e,p in notes_sp2 if VIZ_START<=s<=VIZ_END], 'crimson', 'After merge pass 2')
axes[2].set_title('Stage 6 — After Same-Pitch Merge Pass 2 (final notes)')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('MIDI note')
axes[2].set_yticks(range(57,84,2)); axes[2].set_yticklabels([midi_to_note_name(m) for m in range(57,84,2)])
axes[2].legend(); axes[2].set_xlim(VIZ_START, VIZ_END)
plt.tight_layout(); plt.show()

## 7. (Optional) Snap-to-Beats

When `--snap-to-beats` is enabled, each note's start and end are snapped to the
nearest 16th-note subdivision on the beat grid. This produces tighter rhythmic
quantization at the cost of exact onset timing — the note positions no longer
reflect the precise moment the vocalist sang them.

A guard ensures snapping never collapses a note (new_end − new_start must be
≥ 0.5 × SPLIT_MIN_S). `[source-code] rmvpe.py`

In [ ]:
def snap_to_grid(t, beat_arr, subdivisions=4):
    """Snap time t to the nearest beat subdivision."""
    if len(beat_arr) < 2:
        return t
    idx = np.searchsorted(beat_arr, t)
    lo_idx = max(0, idx - 1)
    hi_idx = min(len(beat_arr) - 1, idx)
    if lo_idx == hi_idx:
        return t
    beat_dur = beat_arr[hi_idx] - beat_arr[lo_idx]
    sub_dur = beat_dur / subdivisions
    grid_start = beat_arr[lo_idx]
    grid = np.arange(grid_start - beat_dur, grid_start + 2 * beat_dur + sub_dur * 0.5, sub_dur)
    return float(grid[np.argmin(np.abs(grid - t))])

notes_snapped = []
for s, e, note in notes_sp2:
    ns = snap_to_grid(s, beat_grid, subdivisions=4)
    ne = snap_to_grid(e, beat_grid, subdivisions=4)
    if ne - ns >= SPLIT_MIN_S * 0.5:
        notes_snapped.append((ns, ne, note))
    else:
        notes_snapped.append((s, e, note))

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ZOOM2_S, ZOOM2_E = 8.0, 14.0
plot_notes(axes[0], [(s,e,p) for s,e,p in notes_sp2 if ZOOM2_S<=s<=ZOOM2_E], 'crimson', 'Before snap')
for bt in beat_grid[(beat_grid >= ZOOM2_S) & (beat_grid <= ZOOM2_E)]:
    axes[0].axvline(bt, color='gray', lw=0.8, alpha=0.4)
axes[0].set_title('Stage 7 — Snap-to-Beats: Before (with beat grid)')
axes[0].set_ylabel('MIDI note')
axes[0].set_yticks(range(57,84,2)); axes[0].set_yticklabels([midi_to_note_name(m) for m in range(57,84,2)])
axes[0].legend(); axes[0].set_xlim(ZOOM2_S, ZOOM2_E)

plot_notes(axes[1], [(s,e,p) for s,e,p in notes_snapped if ZOOM2_S<=s<=ZOOM2_E], 'teal', 'After snap')
for bt in beat_grid[(beat_grid >= ZOOM2_S) & (beat_grid <= ZOOM2_E)]:
    axes[1].axvline(bt, color='gray', lw=0.8, alpha=0.4)
axes[1].set_title('Stage 7 — Snap-to-Beats: After (16th-note grid)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('MIDI note')
axes[1].set_yticks(range(57,84,2)); axes[1].set_yticklabels([midi_to_note_name(m) for m in range(57,84,2)])
axes[1].legend(); axes[1].set_xlim(ZOOM2_S, ZOOM2_E)
plt.tight_layout(); plt.show()

## 8. Summary

Pipeline improvement across all stages, and a multi-panel overview of the full transformation.

In [ ]:
stages = [
    ('Snap + min-dur',       notes_snap,  'steelblue'),
    ('Silence-aware merge',  notes_sil,   'seagreen'),
    ('Same-pitch merge 1',   notes_sp1,   'darkorange'),
    ('Onset split',          notes_split, 'mediumpurple'),
    ('Same-pitch merge 2',   notes_sp2,   'crimson'),
    ('Snap-to-beats',        notes_snapped,'teal'),
]

print(f'{"Stage":<25} {"Notes":>7} {"Unison":>8} {"% Unison":>10}')
print('-' * 55)
for name, notes, _ in stages:
    u = count_unison(notes)
    pct = 100.0 * u / max(len(notes) - 1, 1)
    print(f'{name:<25} {len(notes):>7} {u:>8} {pct:>9.1f}%')

voiced_s = float(np.sum(f0_filtered > 0)) * HOP_S
midi_s = sum(e - s for s, e, _ in notes_sp2)
print(f'\nVoiced audio:  {voiced_s:.1f}s')
print(f'MIDI coverage: {midi_s:.1f}s ({100*midi_s/voiced_s:.1f}% of voiced)')

In [ ]:
PANEL_S, PANEL_E = 8.0, 18.0
fig, axes = plt.subplots(len(stages), 1, figsize=(16, 2.5 * len(stages)), sharex=True)
fig.suptitle('Vocal F0 Pipeline — All Stages Overview', fontsize=13, y=1.01)

for ax, (name, notes, color) in zip(axes, stages):
    u = count_unison(notes)
    plot_notes(ax, [(s,e,p) for s,e,p in notes if PANEL_S<=s<=PANEL_E], color, f'{name} ({len(notes)} notes, {u} unison)')
    ax.set_ylabel('MIDI')
    ax.set_yticks(range(60, 84, 3))
    ax.set_yticklabels([midi_to_note_name(m) for m in range(60, 84, 3)], fontsize=7)
    ax.set_xlim(PANEL_S, PANEL_E)
    ax.legend(loc='upper right', fontsize=8)
    # Draw beat grid
    for bt in beat_grid[(beat_grid >= PANEL_S) & (beat_grid <= PANEL_E)]:
        ax.axvline(bt, color='gray', lw=0.5, alpha=0.25)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()